<a href="https://colab.research.google.com/github/yukisoda1223/forstudy/blob/main/%E3%82%BD%E3%82%B3%E3%82%B9%E3%83%88_%E6%A4%9C%E7%B4%A2%E3%82%A8%E3%83%B3%E3%82%B8%E3%83%B3%E4%BD%9C%E6%88%90_%E3%83%97%E3%83%AD%E3%82%B0%E3%83%A9%E3%83%9F%E3%83%B3%E3%82%B0%E2%85%A0_%E6%9C%80%E7%B5%82%E8%AA%B2%E9%A1%8C.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1, 前準備

In [ ]:
!apt-get -q install mecab libmecab-dev mecab-ipadic-utf8 file

Reading package lists...
Building dependency tree...
Reading state information...
libmecab-dev is already the newest version (0.996-14build9).
mecab-ipadic-utf8 is already the newest version (2.7.0-20070801+main-3).
mecab is already the newest version (0.996-14build9).
file is already the newest version (1:5.41-3ubuntu0.1).
0 upgraded, 0 newly installed, 0 to remove and 52 not upgraded.


In [ ]:
import subprocess
import sys

def install(package):
    subprocess.check_call([sys.executable, "-m", "pip", "install", package])

def apt_install(package):
    subprocess.check_call(["sudo", "apt-get", "install", "-y", package])

# 必要なライブラリのインストール
try:
    # Ubuntu用のMeCabと辞書をインストール
    print("Updating package list...")
    subprocess.check_call(["sudo", "apt-get", "update"])
    print("Installing MeCab and dictionaries...")
    apt_install("mecab")
    apt_install("libmecab-dev")
    apt_install("mecab-ipadic-utf8")
except Exception as e:
    print(f"Error during apt installation: {e}")

# Python用ライブラリをインストール
try:
    print("Installing Python libraries...")
    install("mecab-python3")
    install("ipadic")
    install("flask")
    install("pykakasi")
    install("nltk")
    install("mozcpy")
    # NLTKデータのダウンロード
    import nltk
    nltk.download('wordnet')
    nltk.download('omw-1.4')
except Exception as e:
    print(f"Error during pip installation: {e}")

Updating package list...
Installing MeCab and dictionaries...
Installing Python libraries...


[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
# 必要なライブラリをインストール
!pip install mecab-python3 ipadic pykakasi nltk

# 必要なモジュールをインポート
import MeCab
import pykakasi
import nltk
import csv
from IPython.display import display
import ipywidgets as widgets
import ipadic

# NLTKデータのダウンロード
nltk.download('wordnet')
nltk.download('omw-1.4')
from nltk.corpus import wordnet as wn

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
import csv

# CSVデータを読み込む
data = []
with open("soco_illustrations.csv", encoding="utf-8") as file:
    reader = csv.DictReader(file)
    for row in reader:
        # 空白を除去
        row = {key.strip(): value.strip() for key, value in row.items()}
        # 必要なキーが存在する場合のみ追加
        if "img_url" in row and "title" in row and "post_tag" in row:
            data.append(row)

In [ ]:
from IPython.display import display, HTML

# 2, アプリケーションの実行

In [ ]:
# MeCab初期化
mecab = MeCab.Tagger(ipadic.MECAB_ARGS)

# 名詞抽出用関数
def extract_nouns(text):
    node = mecab.parseToNode(text)
    nouns = []
    while node:
        feature = node.feature.split(",")
        if feature[0] == "名詞":
            nouns.append(node.surface)
        node = node.next
    return nouns  # リストとして返す

# 検索関数（セグメント化）
def search_images(df, queries):
    def count_matches(row):
        # 各行のtitleとpost_tagで一致する単語の数をカウント
        count = sum(any(q in str(val) for val in row[['title', 'post_tag']].values) for q in queries)
        return count

    def is_perfect_match(row):
        # 名詞がすべて一致しているかを判定
        text = " ".join(str(val) for val in row[['title', 'post_tag']].values)
        return all(q in text for q in queries)

    results = df.copy()
    results['matches'] = results.apply(count_matches, axis=1)
    results['perfect_match'] = results.apply(is_perfect_match, axis=1)  # 完全一致フラグ

    # セグメント化: 完全一致 -> 一致数が多い順
    perfect_matches = results[results['perfect_match']].sort_values(by='matches', ascending=False)
    partial_matches = results[~results['perfect_match'] & (results['matches'] > 1)].sort_values(by='matches', ascending=False)  # ★1以下を除外

    return perfect_matches, partial_matches

# 一致度を★で表現
def create_star_rating(matches, max_matches):
    return "★" * matches + "☆" * (max_matches - matches)

def interactive_search(df):
    display(HTML("""
        <div style="text-align:left;">
            <span style="font-size:14px; color:#0075c2;">『ソコスト』検索アプリ</span>
            <span style="font-size:24px; font-weight:bold; color:#0075c2; margin-left:10px;">けんさくん</span>
        </div>
    """))
    query_input = widgets.Text(placeholder="画像のイメージを教えてください")  # 初回のプレースホルダ
    search_button = widgets.Button(description="検索")
    output_area = widgets.Output()
    keywords_area = widgets.Output()

    search_count = 0
    extracted_nouns = []  # 抽出した名詞を保持

    def on_search_click(b):
        nonlocal search_count, extracted_nouns  # 検索回数と名詞リストを保持
        with output_area:
            output_area.clear_output()  # 以前の結果をクリア
            query = query_input.value.strip()
            if not query:
                print("検索キーワードを入力してください。")
                return

            # 名詞を抽出し、保持
            new_nouns = extract_nouns(query)
            if search_count == 0:
                extracted_nouns = new_nouns  # 初回は新しい名詞のみ
            else:
                extracted_nouns.extend(new_nouns)  # 以降は名詞を追加

            # 重複を排除
            extracted_nouns = list(set(extracted_nouns))

            # 検索を実行
            perfect_matches, partial_matches = search_images(df, extracted_nouns)

            # 検索結果を表示
            max_matches = len(extracted_nouns)  # 最大一致数は入力された名詞の数
            if not perfect_matches.empty or not partial_matches.empty:
                print(f"検索結果 ({len(perfect_matches) + len(partial_matches)}件):")

                # 完全一致セグメント
                if not perfect_matches.empty:
                    print("\n【完全一致】")
                    for index, row in perfect_matches.iterrows():
                        star_rating = create_star_rating(row['matches'], max_matches)
                        display(HTML(f"{star_rating} <a href='{row['img_url']}' target='_blank'>{row['title']}</a>"))

                # 部分一致セグメント
                if not partial_matches.empty:
                    print("\n【部分一致】")
                    for index, row in partial_matches.iterrows():
                        star_rating = create_star_rating(row['matches'], max_matches)
                        display(HTML(f"{star_rating} <a href='{row['img_url']}' target='_blank'>{row['title']}</a>"))
            else:
                print("該当する画像が見つかりませんでした。")

            # 検索ボックスを空に
            query_input.value = ""

            # 検索キーワードを表示
            with keywords_area:
                keywords_area.clear_output()
                print("検索キーワード:", ", ".join(extracted_nouns))

            # プレースホルダを変更
            query_input.placeholder = "追加の検索キーワードを入力してください"
            search_count += 1

    # ボタンのクリックイベントを設定
    search_button.on_click(on_search_click)

    # ウィジェットを表示
    display(query_input, search_button, keywords_area, output_area)

# メイン関数
if __name__ == "__main__":
    interactive_search(df)

Text(value='', placeholder='画像のイメージを教えてください')

Button(description='検索', style=ButtonStyle())

Output()

Output()

# 【未実装】3, 表記変換、類義語取得関数

In [ ]:
#表記ゆれ対応

!pip install pykakasi
# pykakasiの初期化
kks = kakasi()


# Mozcpyの初期化
converter = mozcpy.Converter()


# 変換器の設定
def convert_to_hiragana(word):
    conv_result = kks.convert(word)
    return "".join([item['hira'] for item in conv_result])


def convert_to_katakana(word):
    conv_result = kks.convert(word)
    return "".join([item['kana'] for item in conv_result])


def convert_to_kanji(word, n_best=1):
    conv_result = converter.convert(word, n_best=n_best)
    return conv_result


def extract_nouns(text):
    """
    入力されたテキストから名詞を抽出する関数
    """
    node = mecab.parseToNode(text)
    nouns = []
    while node:
        feature = node.feature.split(",")
        if feature[0] == "名詞":
            nouns.append(node.surface)
        node = node.next
    return nouns


def generate_all_combinations(nouns, n_best=1):
    """
    名詞のひらがな・カタカナ・漢字の全ての組み合わせを生成する関数
    """
    all_variants = []
    for noun in nouns:
        hira = convert_to_hiragana(noun)
        kata = convert_to_katakana(noun)
        kanji = convert_to_kanji(hira, n_best=n_best)  # Mozcpyで漢字変換
        all_variants.append([hira, kata] + kanji)


    all_combinations = list(itertools.product(*all_variants))
    return all_combinations



In [ ]:
#類義語取得

import nltk
nltk.download('wordnet')
import nltk
nltk.download('omw-1.4')
from nltk.corpus import wordnet as wn


#類義語を取得するシステム
def get_synonyms(word):
    synonyms = set()
    for synset in wn.synsets(word):
        for lemma in synset.lemmas():
            synonyms.add(lemma.name())
    return list(synonyms)


In [ ]:
import pandas as pd
from IPython.display import display, HTML
import ipywidgets as widgets
import MeCab
import itertools
import nltk
import pykakasi
import mozcpy

In [ ]:

# 必要なダウンロード
nltk.download('wordnet')
nltk.download('omw-1.4')

# MeCab初期化
mecab = MeCab.Tagger(ipadic.MECAB_ARGS)


# pykakasiの初期化
kks = pykakasi.kakasi()

# Mozcpyの初期化
converter = mozcpy.Converter()

# CSVデータ読み込み
df = pd.read_csv("soco_illustrations.csv", encoding="utf-8")


# 表記ゆれ対応の変換関数
def convert_to_hiragana(word):
    conv_result = kks.convert(word)
    return "".join([item['hira'] for item in conv_result])


def convert_to_katakana(word):
    conv_result = kks.convert(word)
    return "".join([item['kana'] for item in conv_result])


def convert_to_kanji(word, n_best=1):
    conv_result = converter.convert(word, n_best=n_best)
    if isinstance(conv_result, str):  # 返り値が文字列の場合
        return [conv_result]
    elif isinstance(conv_result, list):  # 返り値がリストの場合
        return [result['surface'] for result in conv_result]
    else:  # 想定外の形式の場合
        return []


# 名詞抽出関数
def extract_nouns(text):
    node = mecab.parseToNode(text)
    nouns = []
    while node:
        feature = node.feature.split(",")
        if feature[0] == "名詞":
            nouns.append(node.surface)
        node = node.next
    return nouns


# 表記ゆれ生成関数
def generate_all_combinations(nouns, n_best=1):
    all_variants = []
    for noun in nouns:
        hira = convert_to_hiragana(noun)
        kata = convert_to_katakana(noun)
        kanji = convert_to_kanji(hira, n_best=n_best)  # Mozcpyで漢字変換
        all_variants.append([noun, hira, kata] + kanji)

    # すべての組み合わせを生成
    all_combinations = list(itertools.chain(*all_variants))
    return all_combinations


# 類義語取得関数
def get_synonyms(word):
    synonyms = set()
    for synset in wn.synsets(word):
        for lemma in synset.lemmas():
            synonyms.add(lemma.name())
    return list(synonyms)


# 表記ゆれ + 類義語を生成する関数
def generate_expanded_queries(nouns):
    expanded_queries = []
    for noun in nouns:
        # 表記ゆれを取得
        variants = generate_all_combinations([noun])
        # 類義語を取得
        synonyms = get_synonyms(noun)
        # 表記ゆれと類義語を結合
        expanded_queries.extend(variants + synonyms)
    return list(set(expanded_queries))  # 重複を排除


# 検索関数
def search_images(df, queries):
    def count_matches(row):
        # 各行のtitleとpost_tagで一致する単語の数をカウント
        count = sum(any(q in str(val) for val in row[['title', 'post_tag']].values) for q in queries)
        return count

    def is_perfect_match(row):
        # クエリ内の単語がすべて一致しているかを判定
        text = " ".join(str(val) for val in row[['title', 'post_tag']].values)
        return all(q in text for q in queries)

    results = df.copy()
    results['matches'] = results.apply(count_matches, axis=1)
    results['perfect_match'] = results.apply(is_perfect_match, axis=1)  # 完全一致フラグ

    # セグメント化: 完全一致 -> 一致数が多い順
    perfect_matches = results[results['perfect_match']].sort_values(by='matches', ascending=False)
    partial_matches = results[~results['perfect_match'] & (results['matches'] > 1)].sort_values(by='matches', ascending=False)  # ★1以下を除外

    return perfect_matches, partial_matches


# 一致度を★で表現
def create_star_rating(matches, max_matches):
    return "★" * matches + "☆" * (max_matches - matches)


# インタラクティブ検索
def interactive_search(df):
    # タイトルをHTMLで表示
    display(HTML("""
        <div style="text-align:left;">
            <span style="font-size:14px; color:blue;">『ソコスト』検索アプリ</span>
            <span style="font-size:24px; font-weight:bold; color:blue; margin-left:10px;">けんさくん</span>
        </div>
    """))

    query_input = widgets.Text(placeholder="画像のイメージを教えてください")  # 初回のプレースホルダ
    search_button = widgets.Button(description="検索")
    output_area = widgets.Output()
    keywords_area = widgets.Output()

    search_count = 0
    extracted_nouns = []  # 抽出した名詞を保持

    def on_search_click(b):
        nonlocal search_count, extracted_nouns  # 検索回数と名詞リストを保持
        with output_area:
            output_area.clear_output()  # 以前の結果をクリア
            query = query_input.value.strip()
            if not query:
                print("検索キーワードを入力してください。")
                return

            # 名詞を抽出
            nouns = extract_nouns(query)

            # 表記ゆれと類義語を生成
            expanded_queries = generate_expanded_queries(nouns)

            # 検索を実行
            perfect_matches, partial_matches = search_images(df, expanded_queries)

            # 検索結果を表示
            max_matches = len(expanded_queries)  # 最大一致数は入力された名詞の数
            if not perfect_matches.empty or not partial_matches.empty:
                print(f"検索結果 ({len(perfect_matches) + len(partial_matches)}件):")

                # 完全一致セグメント
                if not perfect_matches.empty:
                    print("\n【完全一致】")
                    for index, row in perfect_matches.iterrows():
                        star_rating = create_star_rating(row['matches'], max_matches)
                        display(HTML(f"{star_rating} <a href='{row['img_url']}' target='_blank'>{row['title']}</a>"))

                # 部分一致セグメント
                if not partial_matches.empty:
                    print("\n【部分一致】")
                    for index, row in partial_matches.iterrows():
                        star_rating = create_star_rating(row['matches'], max_matches)
                        display(HTML(f"{star_rating} <a href='{row['img_url']}' target='_blank'>{row['title']}</a>"))
            else:
                print("該当する画像が見つかりませんでした。")

            # 検索ボックスを空に
            query_input.value = ""

            # 検索キーワードを表示
            with keywords_area:
                keywords_area.clear_output()
                print("検索キーワード:", ", ".join(nouns))

            # プレースホルダを変更
            query_input.placeholder = "追加の検索キーワードを入力してください"
            search_count += 1

    # ボタンのクリックイベントを設定
    search_button.on_click(on_search_click)

    # ウィジェットを表示
    display(query_input, search_button, keywords_area, output_area)


# メイン関数
if __name__ == "__main__":
    interactive_search(df)

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


Text(value='', placeholder='画像のイメージを教えてください')

Button(description='検索', style=ButtonStyle())

Output()

Output()